# Mark 2 Shade Design — Bill of Materials

This notebook tracks every component needed for the **Mk 2** shade structure.  
Components are organised into four categories:

| # | Category   | Description |
|---|------------|-------------|
| 1 | **Columns**  | Vertical structural members & footings |
| 2 | **Tensile**  | Webbing, cables, and tensioned supports |
| 3 | **Canopy**   | Shade fabric, ridge & valley elements |
| 4 | **Walls**    | Side enclosures, screens, and panels |

Each category contains one or more **roles** (functional slots in the design).  
Each role may have multiple **design options** (physical alternatives — pick one)  
and each design may have multiple **pricing options** (vendor/package variants — cheapest auto-selected ⭐).

In [120]:
import pandas as pd
from IPython.display import display, Markdown

# ── Master BOM list ──────────────────────────────────────────────
# Each entry: category, role, design, option, sku, qty, unit, unit_price, url
#   • "design"  – groups physical alternatives for a role  (pick ONE)
#   • rows sharing (role, design) are *pricing* variants   (auto-pick cheapest)
bom_rows: list[dict] = []

def _auto_select(df: pd.DataFrame) -> pd.Series:
    """Return a boolean mask marking the single best row per (category, role)."""
    sel = pd.Series(False, index=df.index)
    for (cat, role), rgrp in df.groupby(["category", "role"], sort=False):
        chosen = rgrp["design"].iloc[0]              # first listed design = selected
        dgrp = rgrp[rgrp["design"] == chosen]
        best = dgrp["line_cost"].idxmin()             # cheapest pricing variant
        sel.at[best] = True
    return sel

def show_section(category: str):
    """Display all design × pricing options for *category*; ⭐ = auto-selected."""
    rows = [r for r in bom_rows if r["category"] == category]
    if not rows:
        display(Markdown(f"*No items yet for **{category}**.*"))
        return
    df = pd.DataFrame(rows)
    df["line_cost"] = df["qty"] * df["unit_price"]
    sel = _auto_select(df)

    # Compute total while indices still match the boolean mask
    total = df.loc[sel, "line_cost"].sum()

    df["Pick"] = sel.map({True: "⭐", False: ""})
    df.index = range(1, len(df) + 1)
    df.index.name = "#"
    cols = {
        "role": "Role", "design": "Design", "option": "Option", "sku": "SKU",
        "qty": "Qty", "unit": "Unit", "unit_price": "Unit $",
        "line_cost": "Line $", "Pick": "Pick",
    }
    out = df.rename(columns=cols)[list(cols.values())]
    display(out)
    display(Markdown(f"**Section total (selected): ${total:,.2f}**"))

def get_selected_bom() -> pd.DataFrame:
    """Return only the auto-selected rows across all categories."""
    df = pd.DataFrame(bom_rows)
    if df.empty:
        return df
    df["line_cost"] = df["qty"] * df["unit_price"]
    sel = _auto_select(df)
    return df.loc[sel].reset_index(drop=True)

---
## 0 — Dimensions & Unit Counts
The design is built from modular **ceiling units**. Each unit is a **20 ft × 20 ft** square pyramid with a central peak **3.5 ft** above the base plane (4 triangular canopy panels + webbing).  

- The **first** ceiling unit requires **4 ground columns**; each additional unit shares 2 columns with the previous, adding only **2 new columns**.  
- **Wall panels** are tracked separately — they are cut from canopy tarp waste.

In [121]:
import math

# ── Unit counts ──────────────────────────────────────────────────
CEILING_UNITS = 1    # number of ceiling/canopy modules
WALL_PANELS   = 2    # number of side panels (tracked independently)

# ── Given dimensions (per ceiling unit) ──────────────────────────
ROOF_WIDTH  = 20.0   # ft – side length of the square base
ROOF_DEPTH  = 20.0   # ft – other side length (square, so same)
PEAK_HEIGHT = 3.5    # ft – height of central peak above base plane

# ── Derived: single triangular panel ─────────────────────────────
half_side = ROOF_WIDTH / 2.0                         # 10 ft
slant_height = math.sqrt(half_side**2 + PEAK_HEIGHT**2)  # peak → mid-edge

half_diag = math.sqrt((ROOF_WIDTH / 2)**2 + (ROOF_DEPTH / 2)**2)
slant_edge = math.sqrt(half_diag**2 + PEAK_HEIGHT**2)    # peak → corner

panel_area = ROOF_WIDTH * slant_height / 2.0
total_canopy_area = 4 * panel_area                        # four panels per unit

# ── Derived: webbing length (per ceiling unit) ───────────────────
loop_length = 2 * slant_edge + 2 * ROOF_WIDTH
total_webbing = 2 * loop_length                           # two loops per unit

# ── Derived: column count ────────────────────────────────────────
# First ceiling unit needs 4 columns; each additional shares 2 with
# the previous unit, so only 2 new columns are needed.
n_columns = 4 + 2 * (CEILING_UNITS - 1)   # = 2·N + 2

# ── Derived: total quantities across all ceiling units ───────────
total_webbing_all = total_webbing * CEILING_UNITS
total_tarps       = CEILING_UNITS                         # 1 tarp per unit
total_floating    = CEILING_UNITS                         # 1 floating column per unit

# ── Pipe lot sizing ──────────────────────────────────────────────
pipe_per_column   = 8.0   # ft per ground column
pipe_per_floating = 4.0   # ft per floating column
total_pipe_ft = n_columns * pipe_per_column + total_floating * pipe_per_floating

# ── Cut-sheet layout options (per unit) ──────────────────────────
strip_w = 5 * ROOF_WIDTH / 2
strip_h = slant_height
strip_area = strip_w * strip_h

pin_side = max(ROOF_WIDTH, 2 * slant_height)
pin_area = pin_side ** 2

nested_w    = pin_side
nested_h    = pin_side
nested_area = pin_area
waste_area  = nested_area - total_canopy_area
waste_pct   = waste_area / nested_area * 100

# ── Thread requirements ───────────────────────────────────────────
# Double-row lockstitch around the perimeter of every canopy triangle & wall panel.
WALL_PANEL_W  = ROOF_WIDTH     # ft — wall panel width  (same as roof side)
WALL_PANEL_H  = 12.0           # ft — wall panel height  (from 24×24 tarp cut)
THREAD_ROWS   = 2              # double row of stitching
THREAD_FACTOR = 2.5            # lockstitch thread-to-seam consumption ratio
SPOOL_YARDS   = 400            # est. yards per 1 oz V69 king spool

tri_perim       = ROOF_WIDTH + 2 * slant_edge                  # one canopy triangle
canopy_seam_ft  = 4 * tri_perim * CEILING_UNITS * THREAD_ROWS  # all canopy panels
wall_perim      = 2 * (WALL_PANEL_W + WALL_PANEL_H)            # one wall panel
wall_seam_ft    = wall_perim * WALL_PANELS * THREAD_ROWS       # all wall panels
total_seam_ft   = canopy_seam_ft + wall_seam_ft

canopy_thread_yd = canopy_seam_ft * THREAD_FACTOR / 3
wall_thread_yd   = wall_seam_ft * THREAD_FACTOR / 3
total_thread_yd  = total_seam_ft * THREAD_FACTOR / 3

canopy_spools = math.ceil(canopy_thread_yd / SPOOL_YARDS)
wall_spools   = math.ceil(wall_thread_yd / SPOOL_YARDS)
total_spools  = math.ceil(total_thread_yd / SPOOL_YARDS)

# ── Output ───────────────────────────────────────────────────────
display(Markdown("### Unit counts"))
display(Markdown(
    f"| Parameter | Value |\n"
    f"|-----------|------:|\n"
    f"| Ceiling units | {CEILING_UNITS} |\n"
    f"| Wall panels | {WALL_PANELS} |\n"
    f"| Ground columns (4 + 2 per extra unit) | {n_columns} |\n"
    f"| Floating columns | {total_floating} |\n"
    f"| Tarps needed | {total_tarps} |\n"
    f"| Total pipe | {total_pipe_ft:.0f} lin ft |\n"
))

display(Markdown("### Roof geometry (per unit)"))
display(Markdown(
    f"| Parameter | Value |\n"
    f"|-----------|------:|\n"
    f"| Base footprint | {ROOF_WIDTH:.1f} ft × {ROOF_DEPTH:.1f} ft |\n"
    f"| Peak height | {PEAK_HEIGHT:.1f} ft |\n"
    f"| Slant height (peak → mid-edge) | {slant_height:.2f} ft |\n"
    f"| Slant edge (peak → corner) | {slant_edge:.2f} ft |\n"
))

display(Markdown("### Cut-sheet options (per unit)"))
strip_waste = strip_area - total_canopy_area
strip_wpct  = strip_waste / strip_area * 100
display(Markdown(
    f"| Layout | Sheet | Area | Waste |\n"
    f"|--------|-------|-----:|------:|\n"
    f"| A — Strip ▲▽▲▽ | {strip_w:.1f} ft × {strip_h:.1f} ft | {strip_area:.0f} ft² "
    f"| {strip_waste:.0f} ft² ({strip_wpct:.0f}%) |\n"
    f"| **B — Pinwheel** ⭐ | **{pin_side:.1f} ft × {pin_side:.1f} ft** | **{pin_area:.0f} ft²** "
    f"| **{waste_area:.0f} ft² ({waste_pct:.0f}%)** |\n"
))

display(Markdown("### Webbing"))
display(Markdown(
    f"| Parameter | Value |\n"
    f"|-----------|------:|\n"
    f"| Per unit (2 loops) | {total_webbing:.2f} ft |\n"
    f"| **All {CEILING_UNITS} unit(s)** | **{total_webbing_all:.2f} ft** |\n"
))

display(Markdown("### Thread (double-row lockstitch, ×2.5 consumption)"))
display(Markdown(
    f"| Component | Perimeter | Seam length | Thread | Spools |\n"
    f"|-----------|----------:|------------:|-------:|-------:|\n"
    f"| Canopy triangles ({4 * CEILING_UNITS} pcs) | {tri_perim:.1f} ft ea "
    f"| {canopy_seam_ft:.0f} ft | {canopy_thread_yd:.0f} yd | {canopy_spools} |\n"
    f"| Wall panels ({WALL_PANELS} pcs) | {wall_perim:.1f} ft ea "
    f"| {wall_seam_ft:.0f} ft | {wall_thread_yd:.0f} yd | {wall_spools} |\n"
    f"| **Total** | | **{total_seam_ft:.0f} ft** "
    f"| **{total_thread_yd:.0f} yd** | **{total_spools}** |\n"
))

### Unit counts

| Parameter | Value |
|-----------|------:|
| Ceiling units | 1 |
| Wall panels | 2 |
| Ground columns (4 + 2 per extra unit) | 4 |
| Floating columns | 1 |
| Tarps needed | 1 |
| Total pipe | 36 lin ft |


### Roof geometry (per unit)

| Parameter | Value |
|-----------|------:|
| Base footprint | 20.0 ft × 20.0 ft |
| Peak height | 3.5 ft |
| Slant height (peak → mid-edge) | 10.59 ft |
| Slant edge (peak → corner) | 14.57 ft |


### Cut-sheet options (per unit)

| Layout | Sheet | Area | Waste |
|--------|-------|-----:|------:|
| A — Strip ▲▽▲▽ | 50.0 ft × 10.6 ft | 530 ft² | 106 ft² (20%) |
| **B — Pinwheel** ⭐ | **21.2 ft × 21.2 ft** | **449 ft²** | **25 ft² (6%)** |


### Webbing

| Parameter | Value |
|-----------|------:|
| Per unit (2 loops) | 138.28 ft |
| **All 1 unit(s)** | **138.28 ft** |


### Thread (double-row lockstitch, ×2.5 consumption)

| Component | Perimeter | Seam length | Thread | Spools |
|-----------|----------:|------------:|-------:|-------:|
| Canopy triangles (4 pcs) | 49.1 ft ea | 393 ft | 328 yd | 1 |
| Wall panels (2 pcs) | 64.0 ft ea | 256 ft | 213 yd | 1 |
| **Total** | | **649 ft** | **541 yd** | **2** |


---
## 1 — Columns
Vertical posts, base plates, footings, and associated hardware.  
Primary columns are 8 ft Sch 5 pipe (1.9" OD) with ringlock rosette tops. One 4 ft floating column is cut from the same pipe lot.

In [122]:
# ── Columns ──────────────────────────────────────────────────────
# Pipe lot sized to total_pipe_ft; base collars & footings scale to n_columns.
pipe_price_per_ft = 150.00 / 36  # based on original 36 lin ft lot @ $150
pipe_lot_price = round(total_pipe_ft * pipe_price_per_ft, 2)

# Role: Primary Column  |  Design: Sch 5 Pipe 1.9" OD
bom_rows.append({
    "category": "Columns",
    "role": "Primary Column",
    "design": 'Sch 5 Pipe 1.9" OD',
    "option": f'{n_columns}×8 ft + {total_floating}×4 ft w/ cutting ({total_pipe_ft:.0f} lin ft)',
    "sku": "",
    "qty": 1,
    "unit": f"lot ({total_pipe_ft:.0f} lin ft)",
    "unit_price": pipe_lot_price,
    "url": "https://www.baymetalsca.com/",
})

# Role: Base Collar  |  Design: RingLock Base Collar
bom_rows.append({
    "category": "Columns",
    "role": "Base Collar",
    "design": "RingLock Base Collar",
    "option": "RingLock Starter Base Collar – Hot Dip Galvanized (3.5 lbs)",
    "sku": "U-RLBC",
    "qty": n_columns,
    "unit": "ea",
    "unit_price": 7.75,
    "url": "https://www.scaffoldssupply.com/Ringlock-Base-Collar-p/u-rlbc.htm",
})

# Role: Floating Column  |  Design: Cut from pipe lot
bom_rows.append({
    "category": "Columns",
    "role": "Floating Column",
    "design": "Cut from pipe lot",
    "option": f'4 ft Sch 5 Pipe 1.5" Nom (1.9" OD) – cut from pipe lot above',
    "sku": "",
    "qty": total_floating,
    "unit": "ea (4 ft piece)",
    "unit_price": 0.00,
    "url": "",
})

# Role: Column Footing  |  Design: Wellco Post Base 4-Pack
footing_packs = math.ceil(n_columns / 4)
bom_rows.append({
    "category": "Columns",
    "role": "Column Footing",
    "design": "Wellco Post Base 4-Pack",
    "option": f'1.9" Round Iron Post Base (4-Pack w/ 18 screws)'
              f' — {footing_packs} pack(s) for {n_columns} columns',
    "sku": "RFPBD48P4W18S",
    "qty": footing_packs,
    "unit": "pack (4 pcs)",
    "unit_price": 13.98,
    "url": "https://www.homedepot.com/p/333136963",
})

show_section("Columns")

,Role,Design,Option,SKU,Qty,Unit,Unit $,Line $,Pick
#,,,,,,,,,
1,Primary Column,"Sch 5 Pipe 1.9"" OD",4×8 ft + 1×4 ft w/ cutting (36 lin ft),,1,lot (36 lin ft),150.00,150.00,⭐
2,Base Collar,RingLock Base Collar,RingLock Starter Base Collar – Hot Dip Galvani...,U-RLBC,4,ea,7.75,31.00,⭐
3,Floating Column,Cut from pipe lot,"4 ft Sch 5 Pipe 1.5"" Nom (1.9"" OD) – cut from ...",,1,ea (4 ft piece),0.00,0.00,⭐
4,Column Footing,Wellco Post Base 4-Pack,"1.9"" Round Iron Post Base (4-Pack w/ 18 screws...",RFPBD48P4W18S,1,pack (4 pcs),13.98,13.98,⭐


**Section total (selected): $194.98**

---
## 2 — Tensile
Cables, webbing, turnbuckles, and other tensioned members that brace or support the structure.  
Floating column support uses 1" 4.5K polyester cargo webbing (4,500 lb break strength, ASTM D6775).

In [123]:
# ── Tensile ──────────────────────────────────────────────────────
webbing_ft = math.ceil(total_webbing_all)  # total across all ceiling units

# Role: Floating Column Support
# Design: 1" 4.5K Polyester Webbing  (two PRICING variants — auto-pick cheapest)

#   Pricing A – per linear foot
bom_rows.append({
    "category": "Tensile",
    "role": "Floating Column Support",
    "design": '1" 4.5K Polyester Webbing',
    "option": f'Per Foot (Black) [{webbing_ft} ft for {CEILING_UNITS} unit(s)]',
    "sku": "NS-WEB14500BLK-LF",
    "qty": webbing_ft,
    "unit": "ft",
    "unit_price": 0.29,
    "url": "https://www.uscargocontrol.com/products/1-x-300-black-4-5k-web-no-print-per-linear-foot",
})

#   Pricing B – 300 ft rolls
n_rolls = math.ceil(webbing_ft / 300)
bom_rows.append({
    "category": "Tensile",
    "role": "Floating Column Support",
    "design": '1" 4.5K Polyester Webbing',
    "option": f'Roll 300 ft (Black) [need {webbing_ft} ft → {n_rolls} roll(s)]',
    "sku": "NS-WEB14500BLK",
    "qty": n_rolls,
    "unit": "roll (300 ft)",
    "unit_price": 69.29,
    "url": "https://www.uscargocontrol.com/products/black-polyester-web-1-x-300-3-5k-cargo-webbing",
})

# Role: Webbing-to-Column Hook
# Design: 1" SS Double J Wire Hook  (connects webbing to ringlock base at column top)
n_hooks = 4 * CEILING_UNITS
bom_rows.append({
    "category": "Tensile",
    "role": "Webbing-to-Column Hook",
    "design": '1" SS Double J Wire Hook',
    "option": f'304 SS, 3,000 lb BS / 1,000 lb WLL (0.4 lbs) '
              f'[{n_hooks} for {CEILING_UNITS} unit(s)]',
    "sku": "WHS252",
    "qty": n_hooks,
    "unit": "ea",
    "unit_price": 4.17,
    "url": "https://www.uscargocontrol.com/products/1-long-stainless-steel-wire-hook-3-000lbs",
})

show_section("Tensile")

,Role,Design,Option,SKU,Qty,Unit,Unit $,Line $,Pick
#,,,,,,,,,
1,Floating Column Support,"1"" 4.5K Polyester Webbing",Per Foot (Black) [139 ft for 1 unit(s)],NS-WEB14500BLK-LF,139,ft,0.29,40.31,⭐
2,Floating Column Support,"1"" 4.5K Polyester Webbing",Roll 300 ft (Black) [need 139 ft → 1 roll(s)],NS-WEB14500BLK,1,roll (300 ft),69.29,69.29,
3,Webbing-to-Column Hook,"1"" SS Double J Wire Hook","304 SS, 3,000 lb BS / 1,000 lb WLL (0.4 lbs) [...",WHS252,4,ea,4.17,16.68,⭐


**Section total (selected): $56.99**

---
## 3 — Canopy
Shade fabric, ridge poles, valley cables, and attachment hardware.

In [125]:
# ── Canopy ───────────────────────────────────────────────────────
# Role: Shade Fabric  |  Design: Core Tarps 24×24 ft PE Tarp
bom_rows.append({
    "category": "Canopy",
    "role": "Shade Fabric",
    "design": "Core Tarps 24×24 ft PE Tarp",
    "option": f"Silver/Black 20 mil Heavy-Duty "
              f"(pinwheel cut → 4 panels, {waste_pct:.0f}% waste)",
    "sku": "CT-701-24x24",
    "qty": total_tarps,
    "unit": "ea (24×24 ft)",
    "unit_price": 150.00,
    "url": "https://www.homedepot.com/p/321673909",
})

# Role: Sewing Thread  |  Design: V69 Polyester Thread
bom_rows.append({
    "category": "Canopy",
    "role": "Sewing Thread",
    "design": "V69 Polyester 1 oz King Spool",
    "option": f"Black \u2013 double-row hem on {4 * CEILING_UNITS} triangles "
              f"({canopy_thread_yd:.0f} yd \u2192 {canopy_spools} spool(s))",
    "sku": "T-V",
    "qty": canopy_spools,
    "unit": "spool (~400 yd)",
    "unit_price": 6.49,
    "url": "https://www.seattlefabrics.com/Heavy-Duty-V46-and-V69-Polyester-Thread-on-1-oz-King-Spool_p_860.html",
})

show_section("Canopy")

,Role,Design,Option,SKU,Qty,Unit,Unit $,Line $,Pick
#,,,,,,,,,
1,Shade Fabric,Core Tarps 24×24 ft PE Tarp,Silver/Black 20 mil Heavy-Duty (pinwheel cut →...,CT-701-24x24,1,ea (24×24 ft),150.00,150.00,⭐
2,Sewing Thread,V69 Polyester 1 oz King Spool,Black – double-row hem on 4 triangles (328 yd ...,T-V,1,spool (~400 yd),6.49,6.49,⭐


**Section total (selected): $156.49**

---
## 4 — Walls
Side panels, screens, roll-up walls, and fastening systems.

In [126]:
# ── Walls ────────────────────────────────────────────────────────
# Role: Side Panel  |  Design: Core Tarps 24×24 ft PE Tarp
wall_tarps = math.ceil(WALL_PANELS / 2)

bom_rows.append({
    "category": "Walls",
    "role": "Side Panel",
    "design": "Core Tarps 24×24 ft PE Tarp",
    "option": f"Silver/Black 20 mil Heavy-Duty "
              f"(cut 2× 20 ft × 12 ft panels/tarp) "
              f"— {wall_tarps} tarp(s) for {WALL_PANELS} panels",
    "sku": "CT-701-24x24",
    "qty": wall_tarps,
    "unit": "ea (24×24 ft)",
    "unit_price": 150.00,
    "url": "https://www.homedepot.com/p/321673909",
})

# Role: Sewing Thread  |  Design: V69 Polyester Thread
bom_rows.append({
    "category": "Walls",
    "role": "Sewing Thread",
    "design": "V69 Polyester 1 oz King Spool",
    "option": f"Black \u2013 double-row hem on {WALL_PANELS} panels "
              f"({wall_thread_yd:.0f} yd \u2192 {wall_spools} spool(s))",
    "sku": "T-V",
    "qty": wall_spools,
    "unit": "spool (~400 yd)",
    "unit_price": 6.49,
    "url": "https://www.seattlefabrics.com/Heavy-Duty-V46-and-V69-Polyester-Thread-on-1-oz-King-Spool_p_860.html",
})

show_section("Walls")

,Role,Design,Option,SKU,Qty,Unit,Unit $,Line $,Pick
#,,,,,,,,,
1,Side Panel,Core Tarps 24×24 ft PE Tarp,Silver/Black 20 mil Heavy-Duty (cut 2× 20 ft ×...,CT-701-24x24,1,ea (24×24 ft),150.00,150.00,⭐
2,Sewing Thread,V69 Polyester 1 oz King Spool,Black – double-row hem on 2 panels (213 yd → 1...,T-V,1,spool (~400 yd),6.49,6.49,⭐


**Section total (selected): $156.49**

---
## Full BOM Summary

In [127]:
bom = get_selected_bom()
bom.index = range(1, len(bom) + 1)
bom.index.name = "#"

grand_total = (bom["qty"] * bom["unit_price"]).sum()

# ── Selected Items table ─────────────────────────────────────────
disp_cols = {"category": "Category", "role": "Role", "design": "Design",
             "option": "Option", "sku": "SKU",
             "qty": "Qty", "unit": "Unit", "unit_price": "Unit $",
             "line_cost": "Line $"}
display(Markdown("### Selected items"))
display(bom.rename(columns=disp_cols)[list(disp_cols.values())])
display(Markdown(f"## Grand Total: **${grand_total:,.2f}**"))

# ── Purchase List (grouped by vendor URL) ────────────────────────
display(Markdown("---\n### 🛒 Purchase List"))
purchase = bom[bom["unit_price"] > 0].copy()
purchase["line_cost"] = purchase["qty"] * purchase["unit_price"]
purchase["vendor"] = purchase["url"].apply(
    lambda u: u.split("/")[2] if pd.notna(u) and u else "—")
plist = (purchase.groupby("vendor", sort=False)
         .apply(lambda g: g[["sku", "option", "qty", "unit", "unit_price", "line_cost"]]
                .reset_index(drop=True), include_groups=False)
         )
plist.index.names = ["Vendor", "#"]
pcols = {"sku": "SKU", "option": "Item", "qty": "Qty", "unit": "Unit",
         "unit_price": "Unit $", "line_cost": "Line $"}
display(plist.rename(columns=pcols)[list(pcols.values())])

# ── Normalization / Cost Metrics ─────────────────────────────────
display(Markdown("---\n### 📐 Cost Metrics"))

# Areas
canopy_ft2  = total_canopy_area * CEILING_UNITS       # shade coverage (canopy only)
footprint   = ROOF_WIDTH * ROOF_DEPTH * CEILING_UNITS  # ground footprint
wall_area   = WALL_PANELS * ROOF_WIDTH * pipe_per_column  # rough wall area (width × column height)
total_covered = footprint + wall_area

# Category subtotals
cat_totals = {}
for cat, grp in bom.groupby("category", sort=False):
    cat_totals[cat] = grp["line_cost"].sum()

metrics = [
    ("Grand total",                             f"${grand_total:,.2f}",       ""),
    ("Canopy shade area",                       f"{canopy_ft2:,.0f} ft²",     f"({CEILING_UNITS} unit × {total_canopy_area:,.0f} ft²)"),
    ("Ground footprint",                        f"{footprint:,.0f} ft²",      f"({ROOF_WIDTH:.0f} × {ROOF_DEPTH:.0f} × {CEILING_UNITS})"),
    ("Wall coverage (est.)",                    f"{wall_area:,.0f} ft²",      f"({WALL_PANELS} panels × {ROOF_WIDTH:.0f} ft × {pipe_per_column:.0f} ft)"),
    ("Total shaded + enclosed area",            f"{total_covered:,.0f} ft²",  ""),
    ("**Cost / ft² shade (canopy only)**",      f"**${grand_total / canopy_ft2:,.2f}**", ""),
    ("**Cost / ft² footprint**",                f"**${grand_total / footprint:,.2f}**",   ""),
    ("**Cost / ft² total covered**",            f"**${grand_total / total_covered:,.2f}**", ""),
    ("Cost / ceiling unit",                     f"${grand_total / CEILING_UNITS:,.2f}",   ""),
    ("Cost / wall panel",                       f"${cat_totals.get('Walls', 0) / max(WALL_PANELS, 1):,.2f}", "(walls category only)"),
    ("Structure (Columns) share",               f"${cat_totals.get('Columns', 0):,.2f}",  f"({cat_totals.get('Columns', 0) / grand_total * 100:.0f}%)"),
    ("Tensile share",                           f"${cat_totals.get('Tensile', 0):,.2f}",   f"({cat_totals.get('Tensile', 0) / grand_total * 100:.0f}%)"),
    ("Canopy share",                            f"${cat_totals.get('Canopy', 0):,.2f}",    f"({cat_totals.get('Canopy', 0) / grand_total * 100:.0f}%)"),
    ("Walls share",                             f"${cat_totals.get('Walls', 0):,.2f}",     f"({cat_totals.get('Walls', 0) / grand_total * 100:.0f}%)"),
]

header = "| Metric | Value | Note |\n|--------|------:|------|\n"
rows = "\n".join(f"| {m} | {v} | {n} |" for m, v, n in metrics)
display(Markdown(header + rows))

### Selected items

,Category,Role,Design,Option,SKU,Qty,Unit,Unit $,Line $
#,,,,,,,,,
1,Columns,Primary Column,"Sch 5 Pipe 1.9"" OD",4×8 ft + 1×4 ft w/ cutting (36 lin ft),,1,lot (36 lin ft),150.00,150.00
2,Columns,Base Collar,RingLock Base Collar,RingLock Starter Base Collar – Hot Dip Galvani...,U-RLBC,4,ea,7.75,31.00
3,Columns,Floating Column,Cut from pipe lot,"4 ft Sch 5 Pipe 1.5"" Nom (1.9"" OD) – cut from ...",,1,ea (4 ft piece),0.00,0.00
4,Columns,Column Footing,Wellco Post Base 4-Pack,"1.9"" Round Iron Post Base (4-Pack w/ 18 screws...",RFPBD48P4W18S,1,pack (4 pcs),13.98,13.98
5,Tensile,Floating Column Support,"1"" 4.5K Polyester Webbing",Per Foot (Black) [139 ft for 1 unit(s)],NS-WEB14500BLK-LF,139,ft,0.29,40.31
6,Tensile,Webbing-to-Column Hook,"1"" SS Double J Wire Hook","304 SS, 3,000 lb BS / 1,000 lb WLL (0.4 lbs) [...",WHS252,4,ea,4.17,16.68
7,Canopy,Shade Fabric,Core Tarps 24×24 ft PE Tarp,Silver/Black 20 mil Heavy-Duty (pinwheel cut →...,CT-701-24x24,1,ea (24×24 ft),150.00,150.00
8,Canopy,Sewing Thread,V69 Polyester 1 oz King Spool,Black – double-row hem on 4 triangles (328 yd ...,T-V,1,spool (~400 yd),6.49,6.49
9,Walls,Side Panel,Core Tarps 24×24 ft PE Tarp,Silver/Black 20 mil Heavy-Duty (cut 2× 20 ft ×...,CT-701-24x24,1,ea (24×24 ft),150.00,150.00


## Grand Total: **$564.95**

---
### 🛒 Purchase List

SKU  \
Vendor                  #                      
www.baymetalsca.com     0                      
www.scaffoldssupply.com 0             U-RLBC   
www.homedepot.com       0      RFPBD48P4W18S   
                        1       CT-701-24x24   
                        2       CT-701-24x24   
www.uscargocontrol.com  0  NS-WEB14500BLK-LF   
                        1             WHS252   
www.seattlefabrics.com  0                T-V   
                        1                T-V   

                                                                        Item  \
Vendor                  #                                                      
www.baymetalsca.com     0             4×8 ft + 1×4 ft w/ cutting (36 lin ft)   
www.scaffoldssupply.com 0  RingLock Starter Base Collar – Hot Dip Galvani...   
www.homedepot.com       0  1.9" Round Iron Post Base (4-Pack w/ 18 screws...   
                        1  Silver/Black 20 mil Heavy-Duty (pinwheel cut →...   
                        2  Silver/Black 20 mil Heavy-Duty (cut 2× 20 ft ×...   
www.uscargocontrol.com  0            Per Foot (Black) [139 ft for 1 unit(s)]   
                        1  304 SS, 3,000 lb BS / 1,000 lb WLL (0.4 lbs) [...   
www.seattlefabrics.com  0  Black – double-row hem on 4 triangles (328 yd ...   
                        1  Black – double-row hem on 2 panels (213 yd → 1...   

                           Qty             Unit  Unit $  Line $  
Vendor                  #                                        
www.baymetalsca.com     0    1  lot (36 lin ft)  150.00  150.00  
www.scaffoldssupply.com 0    4               ea    7.75   31.00  
www.homedepot.com       0    1     pack (4 pcs)   13.98   13.98  
                        1    1    ea (24×24 ft)  150.00  150.00  
                        2    1    ea (24×24 ft)  150.00  150.00  
www.uscargocontrol.com  0  139               ft    0.29   40.31  
                        1    4               ea    4.17   16.68  
www.seattlefabrics.com  0    1  spool (~400 yd)    6.49    6.49  
                        1    1  spool (~400 yd)    6.49    6.49

---
### 📐 Cost Metrics

| Metric | Value | Note |
|--------|------:|------|
| Grand total | $564.95 |  |
| Canopy shade area | 424 ft² | (1 unit × 424 ft²) |
| Ground footprint | 400 ft² | (20 × 20 × 1) |
| Wall coverage (est.) | 320 ft² | (2 panels × 20 ft × 8 ft) |
| Total shaded + enclosed area | 720 ft² |  |
| **Cost / ft² shade (canopy only)** | **$1.33** |  |
| **Cost / ft² footprint** | **$1.41** |  |
| **Cost / ft² total covered** | **$0.78** |  |
| Cost / ceiling unit | $564.95 |  |
| Cost / wall panel | $78.25 | (walls category only) |
| Structure (Columns) share | $194.98 | (35%) |
| Tensile share | $56.99 | (10%) |
| Canopy share | $156.49 | (28%) |
| Walls share | $156.49 | (28%) |